In [ ]:
from dataclasses import dataclass, field
from collections import defaultdict
import os

from dotenv import load_dotenv

load_dotenv()  # before project imports so ONTODISCO_LOG_LEVEL is applied correctly

from pymongo.mongo_client import MongoClient
import numpy as np

from src.ontodisco.utils import dedup_base
from src.ontodisco.utils.openai_utils import LLMTripletExtractor


def get_mongo_client(mongo_uri):
    client = MongoClient(mongo_uri)
    return client

In [ ]:
client = get_mongo_client("mongodb://localhost:27018/?directConnection=true")

In [ ]:
db = client.get_database("musique_gpt4_1_mini_onto_triplets")

types = []
for triplet in db.get_collection("initial_triplets").find({}, {"_id": 0, "subject_type": 1, "object_type": 1}):
    types.append(triplet["subject_type"])
    types.append(triplet["object_type"])
types = list(types)
len(types)

79014

In [ ]:
normalizer = dedup_base.normalize_label
all_labels = types
embedder = dedup_base.ContrieverEmbedder(device="cpu")
embed_batch_size = 100
verifier = LLMTripletExtractor(model="Qwen/Qwen3-235B-A22B-Instruct-2507", api_key=os.getenv("AIRI_KEY"), base_url=os.getenv("AIRI_BASE_URL"))

18:05:31 WARNING  huggingface_hub.utils._http — Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

18:05:34 ERROR    src.ontodisco.OpenAIUtils — Unknown model: Qwen/Qwen3-235B-A22B-Instruct-2507. Price will be set to 0.


In [ ]:
normalized_to_raws: dict[str, set[str]] = defaultdict(set)
surface_form_counts: dict[str, int] = defaultdict(int)
count_per_normalized: dict[str, int] = defaultdict(int)

for label in all_labels:
    norm = normalizer(label)
    normalized_to_raws[norm].add(label)
    surface_form_counts[label] += 1
    count_per_normalized[norm] += 1

unique_labels = sorted(normalized_to_raws.keys())


print(f"After normalisation: {len(unique_labels)} unique labels (from {len(set(all_labels))} raw labels)")


After normalisation: 4327 unique labels (from 4333 raw labels)


In [ ]:
texts = unique_labels
embeddings = embedder.embed(texts, batch_size=embed_batch_size)

In [ ]:
cluster_labels = dedup_base.cluster_hac(embeddings, threshold=0.85, linkage='average')

clusters: dict[int, list[str]] = defaultdict(list)
for norm_label, cl in zip(unique_labels, cluster_labels):
    clusters[int(cl)].append(norm_label)

multi_member = sum(1 for m in clusters.values() if len(m) > 1)
print(f"HAC produced {len(clusters)} clusters ({multi_member} with 2+ members, requiring LLM verification)")
print(f"Mean cluster size for clusters with 2+ members: {np.mean([len(v) for v in clusters.values() if len(v) > 1])}")

HAC produced 3938 clusters (353 with 2+ members, requiring LLM verification)
Mean cluster size for clusters with 2+ members: 2.101983002832861


In [ ]:
cluster_labels = dedup_base.cluster_hac(embeddings, threshold=0.75, linkage='average')

clusters: dict[int, list[str]] = defaultdict(list)
for norm_label, cl in zip(unique_labels, cluster_labels):
    clusters[int(cl)].append(norm_label)

multi_member = sum(1 for m in clusters.values() if len(m) > 1)
print(f"HAC produced {len(clusters)} clusters ({multi_member} with 2+ members, requiring LLM verification)")
print(f"Mean cluster size for clusters with 2+ members: {np.mean([len(v) for v in clusters.values() if len(v) > 1])}")

HAC produced 2960 clusters (939 with 2+ members, requiring LLM verification)
Mean cluster size for clusters with 2+ members: 2.45580404685836


In [22]:
for cluster in clusters.values():
    if len(cluster) > 2:
        print(cluster)

['academic discipline', 'medical discipline', 'scientific discipline']
['academic field', 'field of science', 'field of study', 'research field', 'scientific field']
['academic institution', 'education institution', 'educational institution', 'government institution', 'institution']
['academic program', 'academic programs', 'university program']
['academic research topic', 'academic subject', 'academic topic', 'research topic']
['activities', 'activity', 'sport activity']
['administrative body', 'government bodies', 'government body']
['administrative city', 'administrative districts', 'administrative division', 'administrative divisions']
['administrative district', 'regional capital', 'voivodeship']
['administrative entity', 'administrative territorial entity', 'urban territorial entity']
['administrative function', 'government function', 'government functions', 'governmental functions']
['administrative territorial entity class', 'administrative territorial entity type', 'territoria

In [23]:
random_cluster_ids = np.random.choice(list(clusters.keys()), 50)
random_clusters = {_id: clusters[_id] for _id in random_cluster_ids}
for cluster, members in random_clusters.items():
    print(cluster, members)

139 ['city part', 'province part']
2048 ['literary content']
802 ['treaty']
2942 ['political limitation']
53 ['economic and technological development', 'economic development']
31 ['cultural element', 'cultural elements', 'linguistic elements', 'literary elements', 'religious elements']
1892 ['circulation status']
312 ['player category', 'sports player category']
2791 ['fictional character class']
412 ['radio and television show', 'radio broadcast', 'radio program', 'radio show', 'radio shows']
1813 ['criminal allegations']
301 ['number of people', 'quantity of people']
1660 ['report series']
2846 ['government activity']
1698 ['fictional universe']
886 ['legal case', 'legal trial']
227 ['medical procedure', 'procedure']
1091 ['software testing phase']
861 ['opera', 'opera company']
2089 ['commodity']
949 ['political majority']
1999 ['business agreement']
2054 ['historical period subdivision']
269 ['call sign', 'music format', 'radio format', 'radio station format']
597 ['benefit', 'econ

In [24]:
# verified_groups = dedup_base.verify_clusters_with_llm(random_clusters, verifier)

In [33]:
import json
with open("onto_artifacts/verified_groups.json", "r") as f:
    groups = json.load(f)

cluster_lens = []
for cluster in groups:
    cluster_name, members = cluster[0], cluster[1]
    cluster_lens.append(len(members))

cluster_lens = np.array(cluster_lens)
print(np.mean(cluster_lens), min(cluster_lens), max(cluster_lens))
len(cluster_lens)

1.2891602144133412 1 13


3358

In [ ]:
from collections import defaultdict

entity_type_mapping = defaultdict(list)
for cluster in groups:
    cluster_name, members = cluster[0], cluster[1]
    entity_type_mapping[cluster_name] = members
entity_type_mapping


# ------------- 

In [25]:
from src.ontodisco.entity_dedup import deduplicate_entities, collect_entity_surface_forms, _normalize_compound, _parse_compound, CanonicalEntity, EntityDeduplicationResult, _cluster_faiss_nn

12:14:51 DEBUG    faiss.loader — Environment variable FAISS_OPT_LEVEL is not set, so let's pick the instruction set according to the current CPU
12:14:51 INFO     faiss.loader — Loading faiss with AVX2 support.
12:14:51 INFO     faiss.loader — Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
12:14:51 INFO     faiss.loader — Loading faiss.
12:14:51 INFO     faiss.loader — Successfully loaded faiss.


In [26]:
db = client.get_database("musique_gpt4_1_mini_onto_triplets")

triplets = []
for triplet in db.get_collection("initial_triplets").find({}, {"_id": 0, "subject": 1, "object": 1, "subject_type": 1, "object_type": 1}):
    triplets.append(triplet)

In [27]:
def _entity_embedding_text(compound_label: str) -> str:
    name, type_ = _parse_compound(compound_label)
    return f"{name} {type_}" if type_ else name


embedder = dedup_base.ContrieverEmbedder()

def _build_entity(item_id, canonical_label, surface_forms, mention_count,
                    surface_form_counts):
    all_types: set[str] = set()
    for sf in surface_forms:
        _, type_ = _parse_compound(sf)
        if type_:
            all_types.add(type_)
    return CanonicalEntity(
        item_id=item_id,
        canonical_label=canonical_label,
        surface_forms=surface_forms,
        mention_count=mention_count,
        surface_form_counts=surface_form_counts,
        type_labels=all_types,
    )

def _cluster_fn(embeddings: np.ndarray) -> np.ndarray:
        return _cluster_faiss_nn(
            embeddings,
            threshold=0.75,
            top_k=50,
        )

In [28]:
normalized_to_raws: dict[str, set[str]] = defaultdict(set)
surface_form_counts: dict[str, int] = defaultdict(int)
count_per_normalized: dict[str, int] = defaultdict(int)

all_compound_labels = collect_entity_surface_forms(triplets)

for label in all_compound_labels:
    norm = _normalize_compound(label)
    normalized_to_raws[norm].add(label)
    surface_form_counts[label] += 1
    count_per_normalized[norm] += 1

unique_labels = sorted(normalized_to_raws.keys())

print(f"After normalisation: {len(unique_labels)} unique labels (from {len(set(all_compound_labels))} raw labels)")


12:18:59 INFO     src.ontodisco.entity_dedup — Collected 79010 compound entity mentions from 39507 triplets


After normalisation: 33814 unique labels (from 33952 raw labels)


In [29]:
embedding_text_fn = _entity_embedding_text
texts = [embedding_text_fn(norm_label) for norm_label in unique_labels]
embeddings = embedder.embed(texts, batch_size=embed_batch_size)

In [18]:
cluster_fn = _cluster_faiss_nn
cluster_labels = cluster_fn(embeddings)

11:57:14 DEBUG    pymongo.topology — {"message": "Server heartbeat succeeded", "topologyId": {"$oid": "6a3ced24d5b8eee866937653"}, "driverConnectionId": 1, "serverConnectionId": 3695269, "serverHost": "localhost", "serverPort": 27018, "awaited": true, "durationMS": 10006.920039653778, "reply": "{\"topologyVersion\": {\"processId\": {\"$oid\": \"696a21c8e4f311007d4a933f\"}, \"counter\": 6}, \"hosts\": [\"89c7a2ce305c:27017\"], \"setName\": \"89c7a2ce305c\", \"setVersion\": 1, \"isWritablePrimary\": true, \"primary\": \"89c7a2ce305c:27017\", \"me\": \"89c7a2ce305c:27017\", \"electionId\": {\"$oid\": \"7fffffff000000000000000c\"}, \"lastWrite\": {\"opTime\": {\"ts\": {\"$timestamp\": {\"t\": 1782377831, \"i\": 1}}, \"t\": 12}, \"lastWriteDate\": {\"$date\": \"2026-06-25T08:57:11Z\"}, \"majorityOpTime\": {\"ts\": {\"$timestamp\": {\"t\": 1782377831, \"i\": 1}}, \"t\": 12}, \"majorityWriteDate\": {\"$date\": \"2026-06-25T08:57:11Z\"}}, \"maxBsonObjectSize\": 16777216, \"maxMessageSizeBytes\

In [30]:
cluster_labels

array([ 542, 2503, 2224, ...,  891, 1194, 1908], shape=(4327,))

In [31]:
clusters = defaultdict(list) 
for norm_label, cl in zip(unique_labels, cluster_labels):
    clusters[int(cl)].append(norm_label)


In [32]:
for i, cluster in clusters.items():
    if len(cluster) > 2:
        print(cluster)

['"amazing fantasy" #15 [comic book issue]', '270,000 [population count]', '8th fire [documentary series]']
['"amsterdam" [song]', '18:55:48 utc [time]', '18th century [time period]', '8 july 1967 [date]', '8th venice international film festival [film festival]']
['"are you sure hank done it this way" [song]', '1638 [date]', '1678–1684 [time period]', '1985 [year]', '2009 [date]']
['"being happy" [phrase]', '"best thing that ever happened to me" [song]', 'al rbi leader [award]']
['"big girls don\'t cry" [album]', '"big girls don\'t cry" [musical work]', '"can\'t go for that" [creative work]', '8 mile [film]']
['"cos" [nickname]', '"could it be happiness" [song]', 'a&m [organization]']
['"don\'t be cruel" [album]', '1981 national league championship series [sports event]', '1981 voluntary export restraints [trade policy]']
['"don\'t be cruel" [song]', '"elizabeth c. stanton" class transport ship [ship class]', '"envole-moi" [song]', '"every little step" [musical work]']
['"don\'t let it